# StoryForge AI 📝

## 1. Project Overview

**StoryForge AI** is a multi-model AI storytelling tool that uses two Large Language Models (LLMs) working together in an iterative loop:

* **Writer Model** (e.g., OpenAI GPT) creates a story
* **Reviewer Model** (e.g., Anthropic Claude) critiques and gives feedback
* The Writer then **refines the story based on feedback**

The tool supports **multiple languages**, runs for **a minimum of two iterations**, and is exposed through a **Gradio UI**.

This project is designed to:

* Understand LLM orchestration
* Learn tool-calling flows (GPT vs Claude)
* Practice structured prompt design
* Build production-style AI workflows

---

## 2. High-Level Architecture

```
User Input (UI)
   ↓
Story Prompt + Language Selection
   ↓
Writer Model (GPT)
   ↓
Reviewer Model (Claude)
   ↓
Writer Model (Revision)
   ↓
Final Story Output
```

Key concepts used:

* Multi-model coordination
* Iterative feedback loop
* Tool calling (optional)
* Stateless UI with managed conversation state

---

## 3. Core Features

### ✅ Multi-Model Workflow

* GPT generates the story
* Claude reviews the story
* GPT revises the story based on Claude’s feedback

### ✅ Iterative Refinement

* Minimum **2 iterations**
* Can be extended to N iterations

### ✅ Multilingual Support

* Language selected via dropdown
* Prompts dynamically adapted per language

Supported languages (initial):

* English
* Hindi
* German
* French

### ✅ Gradio UI

* Text input for story idea
* Dropdown for language
* Button to start generation
* Chat-style or text output display

In [ ]:
import os
import json
from dotenv import load_dotenv
from scraper import fetch_website_contents, fetch_website_links
from IPython.display import Markdown, display, update_display
from openai import OpenAI 
import gradio as gr
from anthropic import Anthropic
import uuid

In [ ]:
#constants

MODEL_GPT = 'gpt-4.1-nano'
MODEL_CLAUDE = 'claude-sonnet-4-5-20250929'

In [ ]:
load_dotenv(override=True)
openai_api_key= os.getenv('OPENAI_API_KEY')
anthropic_api_key= os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

In [ ]:
#Defining variables for each model
openai = OpenAI()
anthropic = Anthropic()

In [ ]:
# Define the tools

def write_story(topic: str, language: str) -> str:
    """
    Write a short story on a given topic in the selected language.
    """
    return f"Write a short story in {language}. Topic: {topic}"


In [ ]:
def review_story(story: str, language: str) -> str:
    """
    Review a story and provide constructive feedback in the same language.
    """
    return f"Review the following story and give feedback in {language}:\n\n{story}"

In [ ]:
def revise_story(story: str, feedback: str, language: str):
    """
    Revise a story based on feedback and language.
    """
    # Example implementation
    revised = f"Revised Story in {language}:\n\nOriginal Story:\n{story}\n\nBased on Feedback:\n{feedback}"
    return revised


## TOOLS SCHEMAS


In [ ]:
gpt_tools = [
    {
        "type": "function",
        "function": {
            "name": "write_story",
            "description": "Write a short story",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string"},
                    "language": {"type": "string"}
                },
                "required": ["topic", "language"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "revise_story",
            "description": "Revise a story based on feedback",
            "parameters": {
                "type": "object",
                "properties": {
                    "story": {"type": "string"},
                    "feedback": {"type": "string"},
                    "language": {"type": "string"}
                },
                "required": ["story", "feedback", "language"]
            }
        }
    }
]

In [ ]:
claude_tools = [
    {
        "name": "review_story",
        "description": "Review a story and give feedback",
        "input_schema": {
            "type": "object",
            "properties": {
                "story": {"type": "string"},
                "language": {"type": "string"}
            },
            "required": ["story", "language"]
        }
    }
]


## TOOL DISPATCHER

In [ ]:
def handle_tool_call(tool_name, arguments):
    if tool_name == "write_story":
        return write_story(**arguments)

    if tool_name == "review_story":
        return review_story(**arguments)

    if tool_name == "revise_story":
        return revise_story(**arguments)

    raise ValueError(f"Unknown tool: {tool_name}")


## MODEL WRAPPERS

In [ ]:
def ask_gpt(messages, system_prompt, tools=None):
    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[{"role": "system", "content": system_prompt}, *messages],
        tools=tools
    )

    message = response.choices[0].message

    # ✅ CASE 1: GPT calls a tool
    if message.tool_calls:
        tool_call = message.tool_calls[0]

        tool_result = handle_tool_call(
            tool_call.function.name,
            json.loads(tool_call.function.arguments)
        )

        messages.append({
            "role": "assistant",
            "content": tool_result
        })

        followup = openai.chat.completions.create(
            model=MODEL_GPT,
            messages=[{"role": "system", "content": system_prompt}, *messages]
        )

        return followup.choices[0].message.content

    # ✅ CASE 2: Normal GPT response (NO TOOL)
    return message.content


In [ ]:
messages = [
    {"role": "user", "content": "How to learn python"}
]

system_prompt = "You are a story teller. Use the tool `revise_story` to answer it."

print(ask_gpt(messages, system_prompt, tools=gpt_tools))


In [ ]:
def ask_claude(messages, system_prompt, tools=None):
    kwargs = {
        "model": MODEL_CLAUDE,
        "max_tokens": 1024,
        "system": system_prompt,
        "messages": messages,
    }

    if tools:
        kwargs["tools"] = tools

    response = anthropic.messages.create(**kwargs)

    final_text = ""

    for block in response.content:
        # 🔧 TOOL CALL
        if block.type == "tool_use":
            tool_result = handle_tool_call(block.name, block.input)

            messages.append({
                "role": "assistant",
                "content": response.content
            })

            messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": tool_result
                    }
                ]
            })

            followup = anthropic.messages.create(
                model=MODEL_CLAUDE,
                max_tokens=1024,
                system=system_prompt,
                messages=messages
            )

            return followup.content[0].text

        # 📝 NORMAL TEXT
        if block.type == "text":
            final_text += block.text

    return final_text


In [ ]:
messages = [
    {"role": "user", "content": "Give review about python content"}
]

system_prompt = "You are a critical reviewer"

response = ask_claude(
    messages=messages,
    system_prompt=system_prompt,
    tools=claude_tools
)

print(response)


## The Orchestration- How the function should work

In [ ]:
def run_story(topic, language, iterations=1):
    steps = []

    # 1️⃣ Initial story
    messages = [{
        "role": "user",
        "content": f"Write a short story about: {topic}"
    }]

    story = ask_gpt(
        messages=messages,
        system_prompt=f"You are a creative storyteller. Write ONLY in {language}.",
        tools=gpt_tools
    )

    if not isinstance(story, str):
        raise ValueError("Initial story generation failed")

    steps.append(("Story", story))

    for i in range(iterations):

        # 2️⃣ Claude feedback
        feedback_messages = [{
            "role": "user",
            "content": f"""
Review the following story.

Focus on:
- Strengths
- Weaknesses
- Concrete improvement suggestions

Story:
{story}
"""
        }]

        feedback = ask_claude(
            messages=feedback_messages,
            system_prompt=f"""
You are a professional story critic.

STRICT RULES:
- DO NOT ask questions
- DO NOT ask for permission
- DO NOT include conversational text
- DO NOT address the user directly
- Output ONLY the review
- Write the entire response in {language}
"""
        )

        if not isinstance(feedback, str):
            raise ValueError(f"Feedback failed at iteration {i+1}")

        steps.append((f"Feedback {i+1}", feedback))

        # 3️⃣ GPT revision (SAFE)
        revised_story = ask_gpt(
            messages=[
                {
                    "role": "user",
                    "content": f"""
Revise the story using the feedback below.

Feedback:
{feedback}

Original Story:
{story}

RULES:
- Keep the same language: {language}
- Improve clarity, pacing, and emotion
- Output ONLY the revised story
"""
                }
            ],
            system_prompt=f"You are a professional story editor writing in {language}.",
            tools=gpt_tools
        )

        if not isinstance(revised_story, str):
            raise ValueError(f"Revision failed at iteration {i+1}")

        story = revised_story  # ✅ explicit state update
        steps.append((f"Revised Story {i+1}", story))

    return story, steps


In [ ]:
final_story, steps = run_story(
    topic="A hummingbird",
    language="French",
    iterations=3
)

for label, content in steps:
    print(f"\n--- {label} ---\n")
    print(content)


### GRADIO UI WRAPPER

In [ ]:
def gradio_run_story(topic, language, iterations):
    final_story, steps = run_story(
        topic=topic,
        language=language,
        iterations=iterations
    )

    story = ""
    feedback = ""
    revised = final_story

    for label, content in steps:
        if label == "Story":
            story = content
        elif label.startswith("Feedback"):
            feedback += f"\n\n{content}"

    return story, feedback.strip(), revised


In [ ]:
import gradio as gr

with gr.Blocks(title="AI Story Writer & Reviewer") as demo:

    gr.Markdown("## ✨ AI Story Generator, Reviewer & Reviser")

    with gr.Row():
        topic_input = gr.Textbox(
            label="Story Topic",
            placeholder="e.g. A lonely astronaut drifting in space"
        )

    with gr.Row():
        language_dropdown = gr.Dropdown(
            choices=["English", "French", "Spanish", "German", "Hindi"],
            value="English",
            label="Language"
        )

        iterations_slider = gr.Slider(
            minimum=1,
            maximum=3,
            step=1,
            value=1,
            label="Review Iterations"
        )

    generate_btn = gr.Button("🚀 Generate Story")

    with gr.Row():
        story_output = gr.Textbox(
            label="📖 Story (GPT)",
            lines=15
        )

        feedback_output = gr.Textbox(
            label="🧐 Feedback (Claude)",
            lines=15
        )

    revised_output = gr.Textbox(
        label="✍️ Revised Story (GPT)",
        lines=15
    )

    generate_btn.click(
        fn=gradio_run_story,
        inputs=[topic_input, language_dropdown, iterations_slider],
        outputs=[story_output, feedback_output, revised_output]
    )

demo.launch()
